# 09.5 - Pretraining

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Pretraining trains a language model on a large text corpus with a **self-supervised** objective (next-token prediction). This is where the model learns language patterns, world knowledge, and reasoning capacities *before* being adapted for specific tasks. We replicate the essence on a tiny scale.

## 2. Why Does This Matter?

Pretraining explains where LLMs come from and why base models behave a certain way. Understanding it lets you reason about capabilities, data biases, training cost, and why prompting or fine-tuning works.

## 3. Prerequisites

- Unit 09.1 (language models)
- Unit 09.4 (transformers)
- Basic gradient descent

## 4. Learning Objectives

- Explain the next-token-prediction objective
- Train a tiny autoregressive model from scratch
- Observe loss decreasing and generation improving
- Distinguish pretraining from fine-tuning and base from aligned models

## 5. Mental Model

Pretraining is like reading an entire library cover to cover. The model does not 'understand' in the human sense; it learns statistical patterns - which words follow which, how questions relate to answers. The more diverse the 'library', the more versatile the model.

```text
corpus -> token ids -> [t1,t2,t3,t4]
predict t2 from t1, t3 from [t1,t2], t4 from [t1,t2,t3]  (shifted objective)
loss = cross-entropy over all positions -> backprop -> update weights (Adam)
```


## 6. Setup

No pretrained weights are downloaded - we build a tiny model and a tiny synthetic corpus from scratch.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import GPT2Config, GPT2LMHeadModel
torch.manual_seed(0)
np.random.seed(0)
print("Ready.")


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready.


## 7. Build a Tiny Synthetic Corpus + Tokenizer

Real pretraining uses trillions of tokens; we use a handful of short patterns so training is fast on CPU. We build a char-level vocab over the distinct characters.


In [2]:
sentences = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a bird sang in the tree",
    "the fox jumped over the wall",
    "a fish swam under the bridge",
    "the sun rose above the hill",
    "a star shone in the night sky",
    "the rain fell on the roof",
]
corpus = " ".join(sentences)
chars = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
VOCAB = len(chars)
print("Corpus:", corpus)
print("Vocab size:", VOCAB, "| chars:", ''.join(chars))

def encode(s): return [stoi[c] for c in s]
def decode(ids): return ''.join(itos[i] for i in ids if i in itos)

data = np.array(encode(corpus))
print("Encoded corpus length:", len(data))


Corpus: the cat sat on the mat the dog ran in the park a bird sang in the tree the fox jumped over the wall a fish swam under the bridge the sun rose above the hill a star shone in the night sky the rain fell on the roof
Vocab size: 25 | chars:  abcdefghijklmnoprstuvwxy
Encoded corpus length: 212


## 8. Create Training Windows

Slide a window of length `block` over the corpus. Input = window; label = the window shifted by one position (next-token prediction at every position).


In [3]:
block = 24
contexts, targets = [], []
for i in range(0, len(data) - block, block):
    chunk = data[i:i+block+1]
    contexts.append(torch.tensor(chunk[:-1], dtype=torch.long))
    targets.append(torch.tensor(chunk[1:], dtype=torch.long))
print("Number of training windows:", len(contexts))
print("context[0]:", contexts[0].tolist())
print("target[0]: ", targets[0].tolist(), "(shifted by one)")


Number of training windows: 8
context[0]: [19, 8, 5, 0, 3, 1, 19, 0, 18, 1, 19, 0, 15, 14, 0, 19, 8, 5, 0, 13, 1, 19, 0, 19]
target[0]:  [8, 5, 0, 3, 1, 19, 0, 18, 1, 19, 0, 15, 14, 0, 19, 8, 5, 0, 13, 1, 19, 0, 19, 8] (shifted by one)


## 9. Define the Tiny Model and Objective

We use a small GPT-2-style transformer. The training objective in one line: cross-entropy between predicted next-token distribution and the true shifted target at every position.


In [4]:
config = GPT2Config(
    n_layer=2, n_head=2, n_embd=32, vocab_size=VOCAB, n_positions=block,
    bos_token_id=None, eos_token_id=None,
)
model = GPT2LMHeadModel(config)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Objective: average token-level cross-entropy over all positions.")


Model parameters: 27,040
Objective: average token-level cross-entropy over all positions.


## 10. The Pretraining Training Loop

Sample a random window each step, compute loss, backprop, update weights. We log the loss every few steps.


In [5]:
n_steps = 400
losses = []
for step in range(n_steps):
    idx = np.random.randint(0, len(contexts))
    x = contexts[idx].unsqueeze(0)
    y = targets[idx].unsqueeze(0)
    out = model(x, labels=y)
    loss = out.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f"step {step:4d}: loss {loss.item():.4f}")
print(f"\nFinal loss: {losses[-1]:.4f} (started near {losses[0]:.4f})")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


step    0: loss 3.2780


step  100: loss 1.4250


step  200: loss 1.7086


step  300: loss 0.4494



Final loss: 0.5277 (started near 3.2780)


## 11. Plot the Learning Curve

The loss should drop toward a low value as the model learns the corpus statistics.


In [6]:
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross-entropy loss")
plt.title("Pretraining loss over time")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pretrain_curve.png', dpi=100)
print("Saved learning curve to pretrain_curve.png")
print("Loss decreased -> the model captured patterns of the corpus.")


Saved learning curve to pretrain_curve.png
Loss decreased -> the model captured patterns of the corpus.


## 12. Generating After Pretraining

Greedy decode from a start string. A well-trained tiny model should produce plausible continuations (repeated words, common word order).


In [7]:
def greedy_generate(model, start, max_new=20):
    ids = encode(start)
    model.eval()
    with torch.no_grad():
        for _ in range(max_new):
            inp = torch.tensor([ids[-block:]])
            logits = model(inp).logits[:, -1, :]
            nxt = logits.argmax(-1).item()
            ids.append(nxt)
    return decode(ids)

for start in ["the cat", "a bird", "the sun"]:
    print(f"  '{start}' -> '{greedy_generate(model, start)}'")
model.train()


  'the cat' -> 'the catstea nttemtetettetet'


  'a bird' -> 'a birdeun ih u kaoeakkaoea'
  'the sun' -> 'the sunniil nts b rs hnters'


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(25, 32)
    (wpe): Embedding(24, 32)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=96, nx=32)
          (c_proj): Conv1D(nf=32, nx=32)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=128, nx=32)
          (c_proj): Conv1D(nf=32, nx=128)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=32, out_features=25, bias=False)
)

## 13. Failure Case & Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Loss not decreasing | lr too high/low, bad data | tune lr, check data |
| Loss spikes | lr too high, bad batch | reduce lr, skip bad batches |
| Memorized training data | overfit to common examples | dedupe data, regularize |
| Base model gives incomplete answers | predicts continuation, not answers | use aligned model or instruction format |

## 14. Real-World Considerations

- Pretraining large models needs clusters of GPUs, energy, and money (GPT-4: ~13T tokens, $100M+).
- Data quality and deduplication matter far more than raw size.
- Base models must be aligned before they are reliable assistants (see 09.8).
- For most work, use a pretrained model - pretrain only with a compelling reason.

## 15. Common Mistakes

- Confusing pretraining with fine-tuning.
- Assuming base models follow instructions.
- Ignoring data contamination / evaluation leakage.

## 16. When NOT to Pretrain

- When you have limited compute -> use a pretrained model.
- When you only need general text generation -> fine-tune / prompt instead.

## 17. Challenge

Compare the tiny model's loss on windows it was trained on vs on a held-out window, and explain why they differ.


In [8]:
# Held-out: a grammatical but unseen sentence using the same chars
held_out = "the cat sang under the roof"
ho = encode(held_out)
ho_ids = torch.tensor([ho[:block-1]], dtype=torch.long)
ho_lab = torch.tensor([ho[1:block]], dtype=torch.long)
model.eval()
with torch.no_grad():
    held_loss = model(ho_ids, labels=ho_lab).loss.item()
train_loss = sum(losses[-50:]) / 50.0
print(f"Average final training loss: {train_loss:.4f}")
print(f"Held-out unseen sentence loss: {held_loss:.4f}")
model.train()
print("\nLikely outcomes: held-out is higher because the model memorized exact training phrases.")
print("This illustrates generalization vs memorization in language models.")


Average final training loss: 0.4860
Held-out unseen sentence loss: 2.6098

Likely outcomes: held-out is higher because the model memorized exact training phrases.
This illustrates generalization vs memorization in language models.


## 18. Closed-Book Recall

1. What is the training objective for autoregressive models?
2. Why do base models not follow instructions reliably?
3. What is the difference between pretraining and fine-tuning?
4. Why is data quality important for pretraining?

## 19. Teach-Back Questions

Explain to another person:

- The shifted next-token objective in plain words.
- Why a base model is a text predictor, not an assistant.

## 20. Summary

You built a tiny corpus + tokenizer, trained a small GPT-2-like model from scratch with the next-token objective, watched the loss fall, generated continuations, and examined generalization vs memorization.

## 21. Further Experiment

- Train on a larger synthetic corpus (more diverse sentences) and compare generation.
- Try increasing n_embd / n_layer and observe the loss curve change.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, transformers, numpy, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
